# 06: Recommendations

Loads the final checkpoint from `05_expanded_dataset_pipeline.ipynb` and
runs recommendations. This is the fast, presentable notebook -- no CAGR
recomputation, no model training, runs top to bottom in seconds.

## Known limitations
- `predicted_min_cagr` is category-level, not fund-level (see `05` for why).
  Use `min` for fund-specific risk differentiation.
- Recommendations are drawn only from funds with NAV history reaching back
  before 2018, to avoid the short-history bias described in `05`.
- Historical figures reflect the past, not a guarantee of future returns --
  `advise_investment()` shows a balanced / best-case / worst-case range
  rather than a single number for exactly this reason.

## Setup

In [1]:
import pandas as pd
import sys
sys.path.append('../src')
from calculators import compound_interest, recommend_fund, advise_investment

## Load the final checkpoint

Already has: historical mean/min/max CAGR, ML-based `predicted_min_cagr`, readable fund names, and the short-history filter already applied.

In [2]:
risk_return_df_filtered = pd.read_csv('../Data/external/risk_return_df_138funds_filtered_min7yr.csv')

print(risk_return_df_filtered.shape)
print(risk_return_df_filtered['Scheme_Code'].nunique(), "unique funds")
risk_return_df_filtered[['fund', 'window_years', 'mean', 'min', 'predicted_min_cagr']].head()

(744, 9)
75 unique funds


,fund,window_years,mean,min,predicted_min_cagr
0,CANARA ROBECO LARGE CAP FUND - DIRECT PLAN - G...,1,0.165303,-0.201841,-0.097038
1,CANARA ROBECO LARGE CAP FUND - DIRECT PLAN - G...,2,0.158986,-0.052699,-0.010862
2,CANARA ROBECO LARGE CAP FUND - DIRECT PLAN - G...,3,0.154762,0.005373,0.029497
3,CANARA ROBECO LARGE CAP FUND - DIRECT PLAN - G...,4,0.155859,0.052588,0.056677
4,CANARA ROBECO LARGE CAP FUND - DIRECT PLAN - G...,5,0.157720,0.026134,0.063750


## Category spread check

Quick sanity check on how many funds per category survived the short-history filter.

In [3]:
final_selection = pd.read_csv('../Data/external/final_selection_140funds.csv')
category_lookup = final_selection[['Scheme_Code', 'Scheme_Category']].drop_duplicates()

fund_categories = risk_return_df_filtered[['Scheme_Code']].drop_duplicates().merge(
    category_lookup, on='Scheme_Code', how='left'
)
print(fund_categories['Scheme_Category'].value_counts())

Scheme_Category
Equity Scheme - Large Cap Fund                        14
Hybrid Scheme - Aggressive Hybrid Fund                14
Equity Scheme - Mid Cap Fund                          14
Debt Scheme - Short Duration Fund                     13
Debt Scheme - Corporate Bond Fund                     11
Equity Scheme - Small Cap Fund                         8
Income/Debt Oriented Schemes - Corporate Bond Fund     1
Name: count, dtype: int64


## Example recommendations

`advise_investment()` prints a balanced / best-case / worst-case projected value for each recommended fund -- not a single optimistic number -- since the historical average alone reads as a promise rather than a summary of the past.

**Conservative — short horizon, high penalty**

In [4]:
advise_investment(risk_return_df_filtered, principal=100000, years=1, penalty=2.0, risk_column='min', n=1)

Top 1 fund(s) recommened for 1 years at penalty weight 2.0 (risk measure: min):

  #1: ICICI Prudential Short Term Fund - Direct Plan - Growth Option
      Expected Cagr=0.0854 
      Worst Case Cagr return (min)=0.0301
      Score= 0.0854


Projected value of Rs100000 over 1 years, per recommended fund:
  ICICI Prudential Short Term Fund - Direct Plan - Growth Option:
      Balanced estimate:      approx Rs105772.14
      Best case (mean):       approx Rs108537.28
      Worst case (min):   approx Rs103007.01


,fund,mean,min,score
0,ICICI Prudential Short Term Fund - Direct Plan...,0.085373,0.03007,0.085373


**Aggressive — short horizon, low penalty**

In [5]:
advise_investment(risk_return_df_filtered, principal=100000, years=1, penalty=0.2, risk_column='min', n=1)

Top 1 fund(s) recommened for 1 years at penalty weight 0.2 (risk measure: min):

  #1: Nippon India Small Cap Fund - Direct Plan Growth Plan - Bonus Option
      Expected Cagr=0.3142 
      Worst Case Cagr return (min)=-0.3551
      Score= 0.2432


Projected value of Rs100000 over 1 years, per recommended fund:
  Nippon India Small Cap Fund - Direct Plan Growth Plan - Bonus Option:
      Balanced estimate:      approx Rs97954.45
      Best case (mean):       approx Rs131419.93
      Worst case (min):   approx Rs64488.96


,fund,mean,min,score
0,Nippon India Small Cap Fund - Direct Plan Grow...,0.314199,-0.35511,0.243177


**Long horizon — historical vs ML risk measure**

In [6]:
advise_investment(risk_return_df_filtered, principal=100000, years=10, penalty=1.0, risk_column='min', n=3)

Top 3 fund(s) recommened for 10 years at penalty weight 1.0 (risk measure: min):

  #1: Nippon India Small Cap Fund - Direct Plan Growth Plan - Bonus Option
      Expected Cagr=0.2498 
      Worst Case Cagr return (min)=0.1969
      Score= 0.2498

  #2: Kotak Midcap Fund - Direct Plan - Growth
      Expected Cagr=0.2126 
      Worst Case Cagr return (min)=0.1668
      Score= 0.2126

  #3: Axis Small Cap Fund - Direct Plan - Growth
      Expected Cagr=0.2111 
      Worst Case Cagr return (min)=0.1775
      Score= 0.2111


Projected value of Rs100000 over 10 years, per recommended fund:
  Nippon India Small Cap Fund - Direct Plan Growth Plan - Bonus Option:
      Balanced estimate:      approx Rs766692.70
      Best case (mean):       approx Rs929877.29
      Worst case (min):   approx Rs603508.11
  Kotak Midcap Fund - Direct Plan - Growth:
      Balanced estimate:      approx Rs577449.50
      Best case (mean):       approx Rs687126.30
      Worst case (min):   approx Rs467772.70
  Axis

,fund,mean,min,score
0,Nippon India Small Cap Fund - Direct Plan Grow...,0.249806,0.196929,0.249806
1,Kotak Midcap Fund - Direct Plan - Growth,0.212561,0.166819,0.212561
2,Axis Small Cap Fund - Direct Plan - Growth,0.211063,0.177538,0.211063


In [7]:
advise_investment(risk_return_df_filtered, principal=100000, years=10, penalty=1.0, risk_column='predicted_min_cagr', n=3)

Top 3 fund(s) recommened for 10 years at penalty weight 1.0 (risk measure: predicted_min_cagr):

  #1: Nippon India Small Cap Fund - Direct Plan Growth Plan - Bonus Option
      Expected Cagr=0.2498 
      Worst Case Cagr return (predicted_min_cagr)=0.1459
      Score= 0.2498

  #2: Kotak Midcap Fund - Direct Plan - Growth
      Expected Cagr=0.2126 
      Worst Case Cagr return (predicted_min_cagr)=0.1087
      Score= 0.2126

  #3: Axis Small Cap Fund - Direct Plan - Growth
      Expected Cagr=0.2111 
      Worst Case Cagr return (predicted_min_cagr)=0.1459
      Score= 0.2111


Projected value of Rs100000 over 10 years, per recommended fund:
  Nippon India Small Cap Fund - Direct Plan Growth Plan - Bonus Option:
      Balanced estimate:      approx Rs660109.90
      Best case (mean):       approx Rs929877.29
      Worst case (predicted_min_cagr):   approx Rs390342.52
  Kotak Midcap Fund - Direct Plan - Growth:
      Balanced estimate:      approx Rs483869.45
      Best case (mean):  

,fund,mean,predicted_min_cagr,score
0,Nippon India Small Cap Fund - Direct Plan Grow...,0.249806,0.145894,0.249806
1,Kotak Midcap Fund - Direct Plan - Growth,0.212561,0.108692,0.212561
2,Axis Small Cap Fund - Direct Plan - Growth,0.211063,0.145894,0.211063


**Edge cases — guardrails**

In [8]:
# Invalid penalty (outside 0-2) -- should return None cleanly
advise_investment(risk_return_df_filtered, principal=100000, years=5, penalty=3.5, risk_column='min', n=1)

Please enter a valid penalty range


In [9]:
# Out-of-range window_years (only 1-10 exist) -- tests behavior on an empty slice
advise_investment(risk_return_df_filtered, principal=100000, years=15, penalty=1.0, risk_column='min', n=1)

Top 1 fund(s) recommened for 15 years at penalty weight 1.0 (risk measure: min):


Projected value of Rs100000 over 15 years, per recommended fund:


,fund,mean,min,score


## Notes

- **Penalty slider has diminishing effect at longer horizons** in this
  filtered dataset -- most funds' worst-case (`min`) CAGR turns positive by
  mid-length windows, echoing the core Phase 3 finding that downside risk
  shrinks with holding period. Expect penalty to matter more at
  `years=1`-`2` than at `years=10`.
- See `05_expanded_dataset_pipeline.ipynb` for the full derivation of this
  checkpoint (fund selection, CAGR computation, category encoding, quantile
  model, data-quality fixes, short-history filter).